# SAHA — Oral Cancer Screening Model Training Pipeline

**Scientifically valid, fully offline, real-world deployable oral cancer screening.**

## Architecture
- **Base**: EfficientNetB0 (pretrained ImageNet)
- **Output**: 3-class — `Cancer`, `Normal Oral`, `Non-Oral`
- **Calibration**: Temperature scaling (post-hoc)
- **OOD Detection**: Mahalanobis distance on penultimate embeddings
- **Deployment**: INT8 TFLite via quantization-aware training

## Inference Rules
- Non-Oral class dominant → **Reject input**
- Confidence < 0.80 → **Inconclusive**
- Blurry/low-light → **Retake request**
- Never force prediction

## Datasets
1. **Oral Cancer Images** — Kaggle oral cancer dataset (~1300 images, cancer/non-cancer)
2. **Non-Oral Rejection** — 500–1000 images from Intel Image Classification (buildings, forests, streets, etc.)
3. **Additional Healthy** — Open-source oral cavity images for Normal Oral class augmentation

---

⚠️ **Disclaimer**: AI Screening Tool. Not a medical diagnosis. All outputs require clinical confirmation.

## 0. Environment Setup

In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────
!pip install -q tensorflow tensorflow-model-optimization scikit-learn matplotlib seaborn pillow pandas

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_model_optimization as tfmot

from pathlib import Path
from collections import Counter
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import mahalanobis

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Paths
BASE_DIR = Path('/kaggle/working')
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Dataset Assembly & Patient-Level Splitting

### Dataset Sources
1. **Oral Cancer**: Kaggle dataset `ashenafifasilkebede/dataset` or `zaidpy/oral-cancer-lips-and-tongue-images`
2. **Non-Oral (rejection class)**: Kaggle `puneet6060/intel-image-classification` — 1000 random images from buildings/forest/mountain/street
3. **Healthy Oral**: Additional normal mouth images from `rohan0301/dental-diseases-classification` (healthy class)

### Patient-Level Split Strategy
- Images are grouped by patient/folder to prevent data leakage
- 70% train / 15% validation / 15% test (at patient level)
- Stratified to maintain class balance across splits

In [ ]:
# ── 1A. Download & Organize Datasets ──────────────────────────────────────
# On Kaggle, add these datasets to the notebook:
#   1. oral-cancer-lips-and-tongue-images
#   2. intel-image-classification
#
# Adjust paths to match your Kaggle dataset attachment names.

# --- Configuration (adjust these paths to your Kaggle datasets) ---
ORAL_CANCER_DIR = Path('/kaggle/input/oral-cancer-lips-and-tongue-images')
INTEL_IMAGE_DIR = Path('/kaggle/input/intel-image-classification/seg_train/seg_train')

# --- Scan oral cancer dataset ---
def scan_oral_dataset(root_dir):
    """Scan oral cancer dataset and return list of (path, label, patient_id)."""
    records = []
    # Most Kaggle oral cancer datasets have subfolders: cancer / non-cancer
    # or: Normal / OSCC (Oral Squamous Cell Carcinoma)
    for class_dir in sorted(root_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name.lower()
        # Map to our labels
        if any(kw in class_name for kw in ['cancer', 'oscc', 'malignant', 'tumor']):
            label = 'Cancer'
        elif any(kw in class_name for kw in ['normal', 'healthy', 'benign', 'non-cancer']):
            label = 'Normal Oral'
        else:
            print(f'  [SKIP] Unknown class dir: {class_dir.name}')
            continue
        
        for img_path in sorted(class_dir.glob('*')):
            if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp'):
                # Patient ID: use parent folder + first part of filename
                # This groups augmented versions of the same patient image
                patient_id = f'{class_dir.name}_{img_path.stem.split("_")[0].split("-")[0]}'
                records.append({
                    'path': str(img_path),
                    'label': label,
                    'patient_id': patient_id
                })
    return records

oral_records = scan_oral_dataset(ORAL_CANCER_DIR)
print(f'Oral dataset: {len(oral_records)} images')
print(f'  Labels: {Counter(r["label"] for r in oral_records)}')
print(f'  Unique patients: {len(set(r["patient_id"] for r in oral_records))}')

In [ ]:
# ── 1B. Assemble Non-Oral Rejection Images ───────────────────────────────

def sample_non_oral_images(intel_dir, n_samples=800):
    """Sample non-oral images from Intel Image Classification dataset.
    
    Uses buildings, forest, glacier, mountain, sea, street classes.
    These are clearly non-oral, preventing any ambiguity.
    """
    records = []
    non_oral_classes = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
    
    all_images = []
    for cls_name in non_oral_classes:
        cls_dir = intel_dir / cls_name
        if cls_dir.exists():
            imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
            all_images.extend([(str(p), cls_name) for p in imgs])
    
    # Randomly sample
    rng = np.random.RandomState(SEED)
    indices = rng.choice(len(all_images), size=min(n_samples, len(all_images)), replace=False)
    
    for idx in indices:
        path, src_class = all_images[idx]
        # Each image is its own "patient" (no grouping needed for non-oral)
        patient_id = f'nonoral_{Path(path).stem}'
        records.append({
            'path': path,
            'label': 'Non-Oral',
            'patient_id': patient_id
        })
    
    return records

non_oral_records = sample_non_oral_images(INTEL_IMAGE_DIR, n_samples=800)
print(f'Non-oral images sampled: {len(non_oral_records)}')

# ── Combine all records ──────────────────────────────────────────────────
all_records = oral_records + non_oral_records
df = pd.DataFrame(all_records)

print(f'\n=== Full Dataset ===')
print(f'Total images: {len(df)}')
print(f'Class distribution:\n{df["label"].value_counts()}')
print(f'Unique patients: {df["patient_id"].nunique()}')

In [ ]:
# ── 1C. Patient-Level Train/Val/Test Split (70/15/15) ────────────────────

LABEL_MAP = {'Cancer': 0, 'Normal Oral': 1, 'Non-Oral': 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
NUM_CLASSES = 3

df['label_id'] = df['label'].map(LABEL_MAP)

# --- Step 1: Split off 15% test set (at patient level) ---
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_val_idx, test_idx = next(gss_test.split(df, df['label_id'], groups=df['patient_id']))

df_train_val = df.iloc[train_val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

# --- Step 2: Split remaining into 70% train / 15% val ---
# 15/(70+15) ≈ 0.176
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=SEED)
train_idx, val_idx = next(gss_val.split(df_train_val, df_train_val['label_id'], groups=df_train_val['patient_id']))

df_train = df_train_val.iloc[train_idx].reset_index(drop=True)
df_val = df_train_val.iloc[val_idx].reset_index(drop=True)

# --- Verify no data leakage ---
train_patients = set(df_train['patient_id'])
val_patients = set(df_val['patient_id'])
test_patients = set(df_test['patient_id'])

assert len(train_patients & val_patients) == 0, 'LEAKAGE: Train ∩ Val!'
assert len(train_patients & test_patients) == 0, 'LEAKAGE: Train ∩ Test!'
assert len(val_patients & test_patients) == 0, 'LEAKAGE: Val ∩ Test!'
print('✓ Zero data leakage confirmed')

# --- Distribution report ---
for name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f'\n{name}: {len(split_df)} images, {split_df["patient_id"].nunique()} patients')
    print(f'  {dict(split_df["label"].value_counts())}')

## 2. Data Pipeline & Augmentation

In [ ]:
# ── 2. Data Pipeline ──────────────────────────────────────────────────────

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label):
    """Load image, resize to 224x224, normalize to [0,1]."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# ── Data augmentation (training only) ─────────────────────────────────────
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.15, seed=SEED),
    tf.keras.layers.RandomFlip('horizontal', seed=SEED),
    tf.keras.layers.RandomBrightness(0.2, seed=SEED),
    tf.keras.layers.RandomContrast(0.2, seed=SEED),
    tf.keras.layers.RandomZoom(0.1, seed=SEED),
], name='data_augmentation')

def augment(img, label):
    img = data_augmentation(img, training=True)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(dataframe, augment_fn=None, shuffle=False):
    """Create tf.data.Dataset from DataFrame."""
    paths = dataframe['path'].values
    labels = tf.keras.utils.to_categorical(dataframe['label_id'].values, NUM_CLASSES)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

ds_train = make_dataset(df_train, augment_fn=augment, shuffle=True)
ds_val = make_dataset(df_val)
ds_test = make_dataset(df_test)

print(f'Train batches: {tf.data.experimental.cardinality(ds_train).numpy()}')
print(f'Val batches:   {tf.data.experimental.cardinality(ds_val).numpy()}')
print(f'Test batches:  {tf.data.experimental.cardinality(ds_test).numpy()}')

# ── Compute class weights for imbalanced training ─────────────────────────
class_weights_arr = compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2]),
    y=df_train['label_id'].values
)
class_weights = {i: w for i, w in enumerate(class_weights_arr)}
print(f'Class weights: {class_weights}')

## 3. Model Architecture — EfficientNetB0 Transfer Learning

In [ ]:
# ── 3. Model Architecture ─────────────────────────────────────────────────

def build_model(fine_tune_at=None):
    """Build EfficientNetB0 with 3-class head.
    
    Architecture:
        EfficientNetB0 (ImageNet) → GlobalAveragePooling2D → 
        Dropout(0.3) → Dense(128, ReLU) → Dropout(0.2) → Dense(3, softmax)
    
    The Dense(128) layer serves as the embedding layer for OOD detection.
    """
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Freeze base initially
    base_model.trainable = False
    if fine_tune_at is not None:
        base_model.trainable = True
        for layer in base_model.layers[:fine_tune_at]:
            layer.trainable = False
    
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x = tf.keras.layers.Dropout(0.3, name='dropout_1')(x)
    # 128-dim embedding layer — used for Mahalanobis OOD detection
    embeddings = tf.keras.layers.Dense(128, activation='relu', name='embedding')(x)
    x = tf.keras.layers.Dropout(0.2, name='dropout_2')(embeddings)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)
    
    model = tf.keras.Model(inputs, outputs, name='oral_cancer_efficientnetb0')
    return model

model = build_model()
model.summary()
print(f'\nTrainable params: {sum(np.prod(w.shape) for w in model.trainable_weights):,}')
print(f'Non-trainable params: {sum(np.prod(w.shape) for w in model.non_trainable_weights):,}')

## 4. Training Protocol — Warm-up + Fine-tuning with Early Stopping

In [ ]:
# ── 4A. Phase 1: Warm-up (frozen base, train head only) ──────────────────

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', multi_label=False),
    ]
)

print('Phase 1: Warm-up training (base frozen, 5 epochs)...')
warmup_history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=5,
    class_weight=class_weights,
    verbose=1
)

In [ ]:
# ── 4B. Phase 2: Fine-tuning (unfreeze top 30 layers) ────────────────────

# Rebuild with fine-tuning
base = model.get_layer('efficientnetb0')
base.trainable = True
# Freeze all except top 30 layers
fine_tune_from = len(base.layers) - 30
for layer in base.layers[:fine_tune_from]:
    layer.trainable = False

print(f'Fine-tuning from layer {fine_tune_from}/{len(base.layers)}')
print(f'Trainable params after unfreeze: {sum(np.prod(w.shape) for w in model.trainable_weights):,}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', multi_label=False),
    ]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=10,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(OUTPUT_DIR / 'best_model.keras'),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
]

print('\nPhase 2: Fine-tuning (up to 50 epochs, early stopping on val_auc)...')
ft_history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=50,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ── 4C. Training Curves ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(ft_history.history['loss'], label='Train Loss')
axes[0].plot(ft_history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend()

# Accuracy
axes[1].plot(ft_history.history['accuracy'], label='Train Acc')
axes[1].plot(ft_history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend()

# AUROC
axes[2].plot(ft_history.history['auc'], label='Train AUC')
axes[2].plot(ft_history.history['val_auc'], label='Val AUC')
axes[2].set_title('AUROC', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()

## 5. Five-Fold Cross-Validation

In [ ]:
# ── 5. 5-Fold Stratified Group Cross-Validation ──────────────────────────
#
# Stratified by class, grouped by patient_id → no leakage.
# Reports per-fold: sensitivity per class, specificity per class, AUROC.

cv_results = []

# Use train+val for CV, hold out test completely
df_cv = pd.concat([df_train, df_val]).reset_index(drop=True)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(df_cv, df_cv['label_id'], groups=df_cv['patient_id'])):
    print(f'\n══════ Fold {fold+1}/5 ══════')
    
    fold_train = df_cv.iloc[tr_idx]
    fold_val = df_cv.iloc[vl_idx]
    
    # Verify no leakage
    assert len(set(fold_train['patient_id']) & set(fold_val['patient_id'])) == 0
    
    # Build fresh model
    fold_model = build_model()
    fold_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    
    # Quick warm-up
    fold_ds_train = make_dataset(fold_train, augment_fn=augment, shuffle=True)
    fold_ds_val = make_dataset(fold_val)
    
    fold_model.fit(fold_ds_train, epochs=3, verbose=0)  # warm-up
    
    # Unfreeze top layers and fine-tune
    base = fold_model.get_layer('efficientnetb0')
    base.trainable = True
    for layer in base.layers[:len(base.layers)-30]:
        layer.trainable = False
    
    fold_cw = compute_class_weight('balanced', classes=np.array([0,1,2]), y=fold_train['label_id'].values)
    fold_cw = {i: w for i, w in enumerate(fold_cw)}
    
    fold_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    
    fold_model.fit(
        fold_ds_train, validation_data=fold_ds_val,
        epochs=20, class_weight=fold_cw,
        callbacks=[tf.keras.callbacks.EarlyStopping('val_auc', patience=5, mode='max', restore_best_weights=True)],
        verbose=0
    )
    
    # Evaluate
    y_true = fold_val['label_id'].values
    y_pred_probs = fold_model.predict(fold_ds_val, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # Per-class sensitivity & specificity
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    fold_metrics = {'fold': fold + 1}
    
    for c in range(NUM_CLASSES):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - fn - fp
        
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        fold_metrics[f'sensitivity_{INV_LABEL_MAP[c]}'] = sens
        fold_metrics[f'specificity_{INV_LABEL_MAP[c]}'] = spec
    
    # Macro AUROC
    y_true_onehot = tf.keras.utils.to_categorical(y_true, NUM_CLASSES)
    try:
        auroc = roc_auc_score(y_true_onehot, y_pred_probs, multi_class='ovr', average='macro')
    except ValueError:
        auroc = 0.0
    fold_metrics['auroc_macro'] = auroc
    
    cv_results.append(fold_metrics)
    print(f'  AUROC(macro): {auroc:.4f}')
    for c in range(NUM_CLASSES):
        cname = INV_LABEL_MAP[c]
        print(f'  {cname}: Sens={fold_metrics[f"sensitivity_{cname}"]:.3f}, Spec={fold_metrics[f"specificity_{cname}"]:.3f}')
    
    del fold_model
    tf.keras.backend.clear_session()

# ── CV Summary ─────────────────────────────────────────────────────────────
cv_df = pd.DataFrame(cv_results)
print('\n══════ 5-Fold CV Summary ══════')
for col in cv_df.columns:
    if col == 'fold':
        continue
    vals = cv_df[col].values
    print(f'  {col}: {vals.mean():.4f} ± {vals.std():.4f}')

cv_df.to_csv(str(OUTPUT_DIR / 'cv_results.csv'), index=False)
print('\nSaved: cv_results.csv')

## 6. Final Model Evaluation on Held-Out Test Set

In [ ]:
# ── 6A. Evaluate on Test Set ──────────────────────────────────────────────

# Load best model from checkpoint
best_model = tf.keras.models.load_model(str(OUTPUT_DIR / 'best_model.keras'))

y_test_true = df_test['label_id'].values
y_test_probs = best_model.predict(ds_test, verbose=0)
y_test_pred = np.argmax(y_test_probs, axis=1)

# ── Classification Report ─────────────────────────────────────────────────
print('=== Test Set Classification Report ===')
print(classification_report(
    y_test_true, y_test_pred,
    target_names=['Cancer', 'Normal Oral', 'Non-Oral'],
    digits=4
))

# ── AUROC ─────────────────────────────────────────────────────────────────
y_test_onehot = tf.keras.utils.to_categorical(y_test_true, NUM_CLASSES)
auroc_macro = roc_auc_score(y_test_onehot, y_test_probs, multi_class='ovr', average='macro')
print(f'Test AUROC (macro): {auroc_macro:.4f}')

for c in range(NUM_CLASSES):
    auroc_c = roc_auc_score((y_test_true == c).astype(int), y_test_probs[:, c])
    print(f'  {INV_LABEL_MAP[c]} AUROC: {auroc_c:.4f}')

# ── Per-class sensitivity / specificity ────────────────────────────────────
cm = confusion_matrix(y_test_true, y_test_pred, labels=[0, 1, 2])
print(f'\n=== Per-Class Metrics ===')
for c in range(NUM_CLASSES):
    tp = cm[c, c]; fn = cm[c, :].sum() - tp
    fp = cm[:, c].sum() - tp; tn = cm.sum() - tp - fn - fp
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    print(f'  {INV_LABEL_MAP[c]}: Sensitivity={sens:.4f}, Specificity={spec:.4f}')

In [ ]:
# ── 6B. Confusion Matrix Heatmap ─────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cancer', 'Normal Oral', 'Non-Oral'],
            yticklabels=['Cancer', 'Normal Oral', 'Non-Oral'],
            ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Greens',
            xticklabels=['Cancer', 'Normal Oral', 'Non-Oral'],
            yticklabels=['Cancer', 'Normal Oral', 'Non-Oral'],
            ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# ── 6C. Per-Class ROC Curves ─────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c', '#27ae60', '#3498db']

for c in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve((y_test_true == c).astype(int), y_test_probs[:, c])
    auc_val = roc_auc_score((y_test_true == c).astype(int), y_test_probs[:, c])
    ax.plot(fpr, tpr, color=colors[c], lw=2, label=f'{INV_LABEL_MAP[c]} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Per-Class ROC Curves — Oral Cancer Screening', fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'roc_curves.png'), dpi=150)
plt.show()

## 7. OOD Detection — Mahalanobis Distance on Embeddings

In [ ]:
# ── 7. OOD Detection via Mahalanobis Distance ────────────────────────────
#
# Extract 128-dim embeddings from the penultimate 'embedding' layer.
# Compute per-class mean vectors and shared inverse covariance matrix
# on training data. At inference, images with high Mahalanobis distance
# are rejected as out-of-distribution.
#
# Reference: Lee et al. 2018 (NeurIPS) — "A Simple Unified Framework
# for Detecting Out-of-Distribution Samples and Adversarial Attacks"

# Build embedding extractor
embedding_model = tf.keras.Model(
    inputs=best_model.input,
    outputs=best_model.get_layer('embedding').output,
    name='embedding_extractor'
)

# Extract training embeddings
print('Extracting training embeddings for OOD detection...')
ds_train_no_aug = make_dataset(df_train, augment_fn=None, shuffle=False)
train_embeddings = embedding_model.predict(ds_train_no_aug, verbose=1)
train_labels = df_train['label_id'].values

print(f'Embeddings shape: {train_embeddings.shape}')  # (N, 128)

# ── Per-class mean vectors ────────────────────────────────────────────────
class_means = {}
for c in range(NUM_CLASSES):
    mask = train_labels == c
    class_means[c] = train_embeddings[mask].mean(axis=0)
    print(f'Class {INV_LABEL_MAP[c]}: {mask.sum()} samples, mean norm={np.linalg.norm(class_means[c]):.4f}')

# ── Shared covariance matrix ──────────────────────────────────────────────
# Tie covariance across classes (more robust with limited data)
all_centered = []
for c in range(NUM_CLASSES):
    mask = train_labels == c
    all_centered.append(train_embeddings[mask] - class_means[c])

all_centered = np.vstack(all_centered)
cov_matrix = np.cov(all_centered, rowvar=False)

# Regularize to prevent singularity
cov_matrix += np.eye(128) * 1e-6
inv_cov = np.linalg.inv(cov_matrix)

print(f'Covariance matrix shape: {cov_matrix.shape}')
print(f'Condition number: {np.linalg.cond(cov_matrix):.2f}')

In [ ]:
# ── 7B. Compute Mahalanobis distances and find threshold ─────────────────

def compute_mahalanobis(embedding, class_means, inv_cov):
    """Minimum Mahalanobis distance across all classes."""
    min_dist = float('inf')
    for c in range(NUM_CLASSES):
        diff = embedding - class_means[c]
        dist = np.sqrt(diff @ inv_cov @ diff)
        min_dist = min(min_dist, dist)
    return min_dist

# In-distribution distances (training set)
print('Computing in-distribution distances...')
id_distances = np.array([
    compute_mahalanobis(train_embeddings[i], class_means, inv_cov)
    for i in range(len(train_embeddings))
])

# Validation set distances
val_embeddings = embedding_model.predict(ds_val, verbose=0)
val_distances = np.array([
    compute_mahalanobis(val_embeddings[i], class_means, inv_cov)
    for i in range(len(val_embeddings))
])

print(f'In-distribution: mean={id_distances.mean():.2f}, p95={np.percentile(id_distances, 95):.2f}, max={id_distances.max():.2f}')
print(f'Validation: mean={val_distances.mean():.2f}, p95={np.percentile(val_distances, 95):.2f}, max={val_distances.max():.2f}')

# Set threshold at 99th percentile of in-distribution
ood_threshold = np.percentile(id_distances, 99)
print(f'\nOOD Threshold (p99 of in-dist): {ood_threshold:.4f}')

# Visualize
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(id_distances, bins=50, alpha=0.6, label='In-Distribution (Train)', density=True)
ax.hist(val_distances, bins=50, alpha=0.6, label='Validation', density=True)
ax.axvline(ood_threshold, color='red', linestyle='--', lw=2, label=f'OOD Threshold={ood_threshold:.1f}')
ax.set_xlabel('Mahalanobis Distance'); ax.set_ylabel('Density')
ax.set_title('OOD Detection — Mahalanobis Distance Distribution', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'ood_distances.png'), dpi=150)
plt.show()

In [ ]:
# ── 7C. Save OOD artifacts ────────────────────────────────────────────────

# Save as numpy for later use
np.save(str(OUTPUT_DIR / 'ood_class_means.npy'),
        np.array([class_means[c] for c in range(NUM_CLASSES)]))
np.save(str(OUTPUT_DIR / 'ood_inv_covariance.npy'), inv_cov)

ood_config = {
    'threshold': float(ood_threshold),
    'embedding_dim': 128,
    'num_classes': NUM_CLASSES,
    'labels': ['Cancer', 'Normal Oral', 'Non-Oral'],
    'id_mean_distance': float(id_distances.mean()),
    'id_p95_distance': float(np.percentile(id_distances, 95)),
}
with open(OUTPUT_DIR / 'ood_config.json', 'w') as f:
    json.dump(ood_config, f, indent=2)

print('Saved: ood_class_means.npy, ood_inv_covariance.npy, ood_config.json')

# ── Export as Dart constants (for embedding in domain_classifier.dart) ────
def array_to_dart_list(arr, name, indent=2):
    """Convert numpy array to Dart List<double> constant string."""
    flat = arr.flatten().tolist()
    vals = ', '.join(f'{v:.8f}' for v in flat)
    return f'static const {name} = <double>[{vals}];'

dart_lines = [
    '// Auto-generated OOD detection constants from training notebook',
    f'// Generated: {pd.Timestamp.now().isoformat()}',
    f'// OOD threshold: {ood_threshold:.6f}',
    f'static const oodThreshold = {ood_threshold:.6f};',
    f'static const embeddingDim = {128};',
    '',
]
for c in range(NUM_CLASSES):
    dart_lines.append(array_to_dart_list(class_means[c], f'_classMean{c}'))

# Inv covariance is 128x128 = 16384 doubles — store as flat list
dart_lines.append('')
dart_lines.append(f'// Inverse covariance matrix (128x128, row-major)')
dart_lines.append(array_to_dart_list(inv_cov, '_invCovariance'))

with open(OUTPUT_DIR / 'ood_dart_constants.dart', 'w') as f:
    f.write('\n'.join(dart_lines))

print('Saved: ood_dart_constants.dart (embed in domain_classifier.dart)')

## 8. Temperature Scaling — Probability Calibration

In [ ]:
# ── 8. Temperature Scaling (Guo et al. 2017) ─────────────────────────────
#
# Find optimal temperature T that minimizes NLL on validation set.
# Divides logits by T before softmax → calibrated probabilities.

# Extract pre-softmax logits from the model
# Build a logit model (remove softmax)
logit_model = tf.keras.Model(
    inputs=best_model.input,
    outputs=best_model.get_layer('predictions').output
)
# Get weights to compute raw logits
pred_layer = best_model.get_layer('predictions')
W, b = pred_layer.get_weights()

# Get pre-softmax features (input to predictions layer)
pre_pred_model = tf.keras.Model(
    inputs=best_model.input,
    outputs=best_model.get_layer('dropout_2').output
)

val_features = pre_pred_model.predict(ds_val, verbose=0)
val_logits = val_features @ W + b  # Raw logits before softmax
y_val_true = df_val['label_id'].values

def nll_with_temperature(T, logits, labels):
    """Negative log-likelihood with temperature scaling."""
    scaled = logits / T
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    probs = exp_scaled / exp_scaled.sum(axis=1, keepdims=True)
    # Clip for numerical stability
    probs = np.clip(probs, 1e-10, 1.0)
    nll = -np.mean(np.log(probs[np.arange(len(labels)), labels]))
    return nll

# Optimize T
result = minimize_scalar(
    lambda T: nll_with_temperature(T, val_logits, y_val_true),
    bounds=(0.5, 5.0),
    method='bounded'
)
optimal_T = result.x
print(f'Optimal temperature: T = {optimal_T:.4f}')
print(f'NLL before calibration: {nll_with_temperature(1.0, val_logits, y_val_true):.4f}')
print(f'NLL after calibration:  {result.fun:.4f}')

In [ ]:
# ── 8B. Reliability Diagram ───────────────────────────────────────────────

def reliability_diagram(probs, labels, n_bins=10, title=''):
    """Plot reliability diagram and compute ECE."""
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)
    
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs = []
    bin_confs = []
    bin_counts = []
    
    for i in range(n_bins):
        mask = (confidences > bin_edges[i]) & (confidences <= bin_edges[i + 1])
        if mask.sum() > 0:
            bin_accs.append(accuracies[mask].mean())
            bin_confs.append(confidences[mask].mean())
            bin_counts.append(mask.sum())
        else:
            bin_accs.append(0)
            bin_confs.append(0)
            bin_counts.append(0)
    
    # ECE = weighted average of |acc - conf| per bin
    total = sum(bin_counts)
    ece = sum(c * abs(a - f) for a, f, c in zip(bin_accs, bin_confs, bin_counts)) / total if total > 0 else 0
    
    fig, ax = plt.subplots(figsize=(6, 5))
    bin_centers = [(bin_edges[i] + bin_edges[i+1]) / 2 for i in range(n_bins)]
    ax.bar(bin_centers, bin_accs, width=1/n_bins, alpha=0.6, edgecolor='black', label='Observed')
    ax.plot([0, 1], [0, 1], 'r--', lw=2, label='Perfect Calibration')
    ax.set_xlabel('Confidence', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'{title}\nECE = {ece:.4f}', fontweight='bold')
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    
    return fig, ece

# Before calibration
uncal_probs = y_test_probs  # direct softmax output
fig1, ece_before = reliability_diagram(uncal_probs, y_test_true, title='Before Temperature Scaling')
plt.savefig(str(OUTPUT_DIR / 'reliability_before.png'), dpi=150)
plt.show()

# After calibration
test_features = pre_pred_model.predict(ds_test, verbose=0)
test_logits = test_features @ W + b
scaled_logits = test_logits / optimal_T
exp_scaled = np.exp(scaled_logits - scaled_logits.max(axis=1, keepdims=True))
cal_probs = exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

fig2, ece_after = reliability_diagram(cal_probs, y_test_true, title='After Temperature Scaling')
plt.savefig(str(OUTPUT_DIR / 'reliability_after.png'), dpi=150)
plt.show()

print(f'\nECE before: {ece_before:.4f}')
print(f'ECE after:  {ece_after:.4f}')
print(f'Temperature: {optimal_T:.4f}')

## 9. Quantization-Aware Training & TFLite Export

In [ ]:
# ── 9A. Quantization-Aware Training ───────────────────────────────────────

print('Applying quantization-aware training...')

# Clone the best model and apply QAT
qat_model = tfmot.quantization.keras.quantize_model(best_model)

qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print('Fine-tuning with QAT (5 epochs)...')
qat_history = qat_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=5,
    class_weight=class_weights,
    verbose=1
)

# Evaluate QAT model
y_qat_probs = qat_model.predict(ds_test, verbose=0)
y_qat_pred = np.argmax(y_qat_probs, axis=1)
qat_auroc = roc_auc_score(
    tf.keras.utils.to_categorical(y_test_true, NUM_CLASSES),
    y_qat_probs, multi_class='ovr', average='macro'
)
print(f'\nQAT Model AUROC: {qat_auroc:.4f} (vs FP32: {auroc_macro:.4f})')
print(f'AUROC drop: {(auroc_macro - qat_auroc) * 100:.2f}%')

In [ ]:
# ── 9B. Convert to TFLite INT8 ───────────────────────────────────────────

def representative_dataset():
    """Generator for full integer quantization calibration."""
    for batch_imgs, _ in ds_train.take(20):
        for img in batch_imgs:
            yield [tf.expand_dims(img, 0)]

# Convert QAT model
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.float32  # Keep output as float for probability

print('Converting to TFLite INT8...')
tflite_model = converter.convert()

tflite_path = OUTPUT_DIR / 'oral_cancer_efficientnetb0.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f'TFLite model saved: {tflite_path}')
print(f'Model size: {model_size_mb:.2f} MB')

# ── SHA-256 checksum (for model governance) ───────────────────────────────
sha256_hash = hashlib.sha256(tflite_model).hexdigest()
print(f'SHA-256: {sha256_hash}')

In [ ]:
# ── 9C. Validate TFLite Accuracy ─────────────────────────────────────────

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f'Input: {input_details[0]["shape"]}, dtype={input_details[0]["dtype"]}')
print(f'Output: {output_details[0]["shape"]}, dtype={output_details[0]["dtype"]}')

# Run inference on test set
tflite_preds = []
for batch_imgs, _ in ds_test:
    for img in batch_imgs:
        # Quantize input if needed
        if input_details[0]['dtype'] == np.uint8:
            input_scale, input_zp = input_details[0]['quantization']
            img_uint8 = (img.numpy() / input_scale + input_zp).astype(np.uint8)
            input_data = np.expand_dims(img_uint8, axis=0)
        else:
            input_data = np.expand_dims(img.numpy(), axis=0).astype(np.float32)
        
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        tflite_preds.append(output[0])

tflite_preds = np.array(tflite_preds)
tflite_pred_labels = np.argmax(tflite_preds, axis=1)

# Compare
tflite_auroc = roc_auc_score(
    tf.keras.utils.to_categorical(y_test_true, NUM_CLASSES),
    tflite_preds, multi_class='ovr', average='macro'
)

print(f'\n=== TFLite INT8 Validation ===')
print(f'FP32 AUROC:    {auroc_macro:.4f}')
print(f'TFLite AUROC:  {tflite_auroc:.4f}')
print(f'Accuracy drop: {(auroc_macro - tflite_auroc) * 100:.2f}%')

if abs(auroc_macro - tflite_auroc) < 0.03:
    print('✓ Quantization accuracy drop < 3% — PASS')
else:
    print('✗ WARNING: Quantization accuracy drop > 3%')

# Benchmark inference time
import time
times = []
sample_img = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
if input_details[0]['dtype'] == np.uint8:
    input_scale, input_zp = input_details[0]['quantization']
    sample_img = (sample_img / input_scale + input_zp).astype(np.uint8)

for _ in range(50):
    t0 = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], sample_img)
    interpreter.invoke()
    times.append(time.perf_counter() - t0)

print(f'\nInference benchmark (50 runs):')
print(f'  Mean: {np.mean(times)*1000:.1f} ms')
print(f'  P95:  {np.percentile(times, 95)*1000:.1f} ms')

## 10. Export Summary & Deployment Artifacts

In [ ]:
# ── 10. Final Summary & Export ────────────────────────────────────────────

summary = {
    'model_name': 'oral_cancer_efficientnetb0',
    'version': '2.0.0',
    'architecture': 'EfficientNetB0 + Dense(128) + Dense(3)',
    'input_shape': [1, 224, 224, 3],
    'output_classes': ['Cancer', 'Normal Oral', 'Non-Oral'],
    'quantization': 'INT8 (QAT)',
    'model_size_mb': round(model_size_mb, 2),
    'sha256': sha256_hash,
    'temperature': round(float(optimal_T), 4),
    'ood_threshold': round(float(ood_threshold), 4),
    'inference_rules': {
        'reject_if_non_oral_above': 0.50,
        'inconclusive_if_confidence_below': 0.80,
    },
    'metrics': {
        'test_auroc_macro': round(float(auroc_macro), 4),
        'tflite_auroc_macro': round(float(tflite_auroc), 4),
        'quantization_drop_pct': round(float(auroc_macro - tflite_auroc) * 100, 2),
        'ece_before_calibration': round(float(ece_before), 4),
        'ece_after_calibration': round(float(ece_after), 4),
    },
    'cv_auroc_mean': round(float(cv_df['auroc_macro'].mean()), 4),
    'cv_auroc_std': round(float(cv_df['auroc_macro'].std()), 4),
    'datasets': [
        'Kaggle: oral-cancer-lips-and-tongue-images',
        'Kaggle: intel-image-classification (non-oral rejection)',
    ],
    'train_samples': len(df_train),
    'val_samples': len(df_val),
    'test_samples': len(df_test),
    'disclaimer': 'AI Screening Tool. Not a medical diagnosis.',
}

with open(OUTPUT_DIR / 'model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

print('\n=== Output Files ===')
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size
    print(f'  {f.name}: {size/1024:.1f} KB')

print('\n✓ Training pipeline complete.')
print('  Copy oral_cancer_efficientnetb0.tflite → assets/models/')
print('  Copy OOD constants → domain_classifier.dart')
print('  Set temperature T in cancer_classifier.dart')

## 10B. TensorFlow.js Export (for SAHA Web)

Exports the trained Keras model to TF.js LayersModel format for browser-based
inference. The output `model.json` + weight shard `.bin` files should be copied
to `web/models/oral_cancer/` in the Flutter project.

In [ ]:
# ── TF.js Export ──────────────────────────────────────────────────────────
!pip install -q tensorflowjs
import tensorflowjs as tfjs

tfjs_output = OUTPUT_DIR / 'tfjs_oral_cancer'
tfjs.converters.save_keras_model(best_model, str(tfjs_output))

print(f'\n=== TF.js Model Files ===')
for f in sorted(tfjs_output.iterdir()):
    print(f'  {f.name}: {f.stat().st_size/1024:.1f} KB')

tfjs_total = sum(f.stat().st_size for f in tfjs_output.iterdir())
print(f'  Total: {tfjs_total/1024:.1f} KB')
print('\n✓ TF.js export complete.')
print('  Copy contents of outputs/tfjs_oral_cancer/ → web/models/oral_cancer/')

## Kaggle Execution Instructions

1. **Create a new Kaggle Notebook** at https://www.kaggle.com/code
2. **Add Datasets** (right panel → "+ Add Data"):
   - `oral-cancer-lips-and-tongue-images`
   - `intel-image-classification`
3. **Enable GPU** (Settings → Accelerator → GPU T4 x2)
4. **Upload** this notebook or paste all cells
5. **Click "Run All"** — takes ~30-45 minutes
6. **Download outputs** from the `/kaggle/working/outputs/` folder:
   - `oral_cancer_efficientnetb0.tflite` → `assets/models/`
   - `tfjs_oral_cancer/` folder contents → `web/models/oral_cancer/`
   - `model_summary.json` → note the `temperature` and `sha256` values
   - `audio_norm_stats.json` → not needed for oral cancer
7. **Update Dart code**:
   - Set `_temperature` in `cancer_classifier.dart` to the value from `model_summary.json`
   - Set SHA-256 in `model_governance.dart` model registry

## Testing Checklist for Real-World Validation

| # | Test | Expected Result | Status |
|---|------|-----------------|--------|
| 1 | Feed cancer image | Cancer class with p > 0.80 | ☐ |
| 2 | Feed healthy oral | Normal Oral with p > 0.80 | ☐ |
| 3 | Feed car/building | Non-Oral dominant → **REJECT** | ☐ |
| 4 | Feed blurry oral | Low quality score → **Retake** | ☐ |
| 5 | Feed dark/underexposed | Low quality → **Retake** | ☐ |
| 6 | Feed ambiguous oral | Confidence < 0.80 → **Inconclusive** | ☐ |
| 7 | Feed random noise | OOD distance > threshold → **Reject** | ☐ |
| 8 | Inference time < 5s | Benchmark on target device | ☐ |
| 9 | Model size < 10 MB | Check .tflite file size | ☐ |
| 10 | Quantization drop < 3% | Compare FP32 vs INT8 AUROC | ☐ |
| 11 | ECE < 0.05 after calibration | Check reliability diagram | ☐ |
| 12 | 5-fold CV std < 0.05 | Check CV results table | ☐ |

---

⚠️ **This is a screening tool, not a diagnostic device.**

All positive results MUST be confirmed by a qualified healthcare professional.